Query Execution plan

In [0]:
%sql
EXPLAIN
SELECT
  DATE(event_time) AS event_date,
  SUM(price) AS total_sales
FROM ecommerce.events_delta
WHERE event_type = 'purchase'
GROUP BY DATE(event_time);


plan
"== Physical Plan == AdaptiveSparkPlan isFinalPlan=false +- == Initial Plan == ColumnarToRow +- PhotonResultStage +- PhotonGroupingAgg(keys=[_groupingexpression#13196], functions=[finalmerge_sum(merge sum#13224) AS sum(price)#13194]) +- PhotonShuffleExchangeSource +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#8356] +- PhotonShuffleExchangeSink hashpartitioning(_groupingexpression#13196, 1024) +- PhotonGroupingAgg(keys=[_groupingexpression#13196], functions=[partial_sum(price#13191) AS sum#13224]) +- PhotonProject [price#13191, cast(event_time#13185 as date) AS _groupingexpression#13196] +- PhotonScan parquet workspace.ecommerce.events_delta[event_time#13185,event_type#13186,price#13191] DataFilters: [isnotnull(event_type#13186), (event_type#13186 = purchase)], DictionaryFilters: [(event_type#13186 = purchase)], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-cd6ms/uc/3a074064-ac81-4a4d-96fa-abd28b8ef3c1..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct, RequiredDataFilters: [isnotnull(event_type#13186), (event_type#13186 = purchase)] == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = events_delta"


Partition large tables

In [0]:
%sql
CREATE TABLE ecommerce.events_partitioned
USING DELTA
PARTITIONED BY (event_date)
AS
SELECT
  *,
  DATE(event_time) AS event_date
FROM ecommerce.events_delta;


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8935771153842098>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE TABLE ecommerce.events_partitioned\nUSING DELTA\nPARTITIONED BY (event_date)\nAS\nSELECT\n  *,\n  DATE(event_time) AS event_date\nFROM ecommerce.events_delta;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic

In [0]:
%sql
SELECT *
FROM ecommerce.events_partitioned
LIMIT 20;



event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,event_date
2019-11-08T13:11:03.000Z,view,11200239,2053013562946749253,appliances.personal.scales,elenberg,7.7,353826654,a592ebd7-881f-4b66-8b78-2409e18366fa,2019-11-08
2019-11-08T06:43:40.000Z,view,15901730,2053013558190408249,null,granhel,42.21,381213163,09e706ce-3a05-4289-aa6e-ff0ee943e388,2019-11-08
2019-11-08T03:58:29.000Z,view,4804718,2053013554658804075,electronics.audio.headphone,apple,360.06,383549276,49e056af-0ca4-49d6-8321-f78df44ca3eb,2019-11-08
2019-11-08T17:55:04.000Z,view,1003769,2053013555631882655,electronics.smartphone,huawei,510.41,428898875,c482e0b2-257d-41a1-86f4-3b166d1b992e,2019-11-08
2019-11-08T08:14:08.000Z,view,1004625,2053013555631882655,electronics.smartphone,fly,43.73,442208833,a489010c-e845-4ac7-a418-174b88cfdffd,2019-11-08
2019-11-08T04:42:59.000Z,view,28712327,2053013565639492569,apparel.shoes,baden,64.61,457801986,6130a904-6507-49e3-a1f8-34bcc7f39bbb,2019-11-08
2019-11-08T19:22:52.000Z,view,7901064,2053013556487520725,furniture.kitchen.chair,shenma,56.63,466757603,eec1d367-a73c-4d9d-989a-c4a5ffae26ff,2019-11-08
2019-11-08T12:49:07.000Z,view,12703353,2053013553559896355,null,nokian,124.33,470457250,c698c420-f4a1-4cf3-9fa8-44cc742383b9,2019-11-08
2019-11-08T04:37:10.000Z,view,26400219,2053013563651392361,null,null,251.49,478988170,6ea4009e-8198-4b4c-b410-338558e0b534,2019-11-08
2019-11-08T16:02:30.000Z,view,1003310,2053013555631882655,electronics.smartphone,apple,715.96,486758009,92df9a02-168f-4675-b627-31e4595a6271,2019-11-08


In [0]:
%sql
OPTIMIZE ecommerce.events_partitioned
ZORDER BY (user_id, event_type);


path,metrics
,"List(23, 8, List(6022257, 104936061, 4.335210417391305E7, 23, 997098396), List(67906721, 225042994, 1.29943728625E8, 8, 1039549829), 30, List(minCubeSize(107374182400), List(0, 0), List(30, 2299357323), 0, List(8, 1039549829), 8, null), null, 0, 1, 30, 22, false, 0, 0, 1768757644821, 1768757681725, 8, 8, null, List(0, 0), null, 10, 10, 66155, 0, null)"


In [0]:
%sql
SELECT COUNT(*)
FROM ecommerce.events_delta
WHERE event_type = 'purchase'
  AND DATE(event_time) = '2024-01-10';


COUNT(*)
0


In [0]:
%sql 
SELECT COUNT(*)
FROM ecommerce.events_partitioned
WHERE event_date = '2024-01-10'
  AND event_type = 'purchase';


COUNT(*)
0
